#  Google Play Store Intelligence Dashboard

# Data Cleaning

## Overview

Raw datasets often contain missing values, duplicate records, incorrect data types, and inconsistent formatting.

The objective of this notebook is to clean the Google Play Store dataset and prepare it for feature engineering and analysis.

---

#  Objectives

In this notebook we will:

- Create a working copy of the dataset
- Remove corrupted records
- Remove duplicate rows
- Handle missing values
- Convert columns into appropriate data types
- Prepare a clean dataset for analysis

#  Import Libraries

In [31]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

#  Load Dataset

In [32]:
apps = pd.read_csv("../data/raw/googleplaystore.csv")
reviews = pd.read_csv("../data/raw/googleplaystore_user_reviews.csv")

apps_clean = apps.copy()
reviews_clean = reviews.copy()

# Dataset Overview Before Cleaning

In [33]:
print("Apps Shape :", apps_clean.shape)
print("Reviews Shape :", reviews_clean.shape)

Apps Shape : (10841, 13)
Reviews Shape : (64295, 5)


#  Remove Corrupted Record

One record has a rating greater than 5 due to incorrect column alignment.

Since app ratings should only range between 1 and 5, this record is removed.

In [34]:
apps_clean = apps_clean[apps_clean["Rating"] <= 5]

In [35]:
apps_clean["Rating"].max()

np.float64(5.0)

# Remove Duplicate Rows

In [36]:
print("Before:", apps_clean.shape)

apps_clean = apps_clean.drop_duplicates()

print("After:", apps_clean.shape)

Before: (9366, 13)
After: (8892, 13)


# Handle Missing Values

Let's inspect missing values before deciding how to treat them.

In [37]:
apps_clean.isnull().sum().sort_values(ascending=False)

Current Ver       4
Android Ver       2
Rating            0
Category          0
App               0
Size              0
Reviews           0
Installs          0
Type              0
Content Rating    0
Price             0
Last Updated      0
Genres            0
dtype: int64

### Missing Rating

Ratings are missing for some applications.

Instead of removing the rows, we keep them since ratings are unavailable rather than incorrect.

# Convert Reviews to Numeric

In [38]:
apps_clean["Reviews"] = pd.to_numeric(
    apps_clean["Reviews"]
)

In [39]:
apps_clean["Reviews"].dtype

dtype('int64')

# Clean Install Counts

The Installs column contains commas and '+' symbols.

These characters are removed before converting the column into numeric format.

In [40]:
apps_clean["Installs"] = (
    apps_clean["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

apps_clean["Installs"] = pd.to_numeric(
    apps_clean["Installs"]
)

In [41]:
apps_clean["Installs"].head()

0       10000
1      500000
2     5000000
3    50000000
4      100000
Name: Installs, dtype: int64

# Clean Price Column

The Price column contains '$' symbols.

These symbols are removed before converting the column into numeric format.

In [42]:
apps_clean["Price"] = (
    apps_clean["Price"]
    .str.replace("$", "", regex=False)
)

apps_clean["Price"] = pd.to_numeric(
    apps_clean["Price"]
)

In [43]:
apps_clean["Price"].head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Price, dtype: float64

# Convert Last Updated to Datetime

In [44]:
apps_clean["Last Updated"] = pd.to_datetime(
    apps_clean["Last Updated"]
)

In [45]:
apps_clean["Last Updated"].dtype

dtype('<M8[us]')

# Verify Cleaned Dataset

In [46]:
apps_clean.info()

<class 'pandas.DataFrame'>
Index: 8892 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             8892 non-null   str           
 1   Category        8892 non-null   str           
 2   Rating          8892 non-null   float64       
 3   Reviews         8892 non-null   int64         
 4   Size            8892 non-null   str           
 5   Installs        8892 non-null   int64         
 6   Type            8892 non-null   str           
 7   Price           8892 non-null   float64       
 8   Content Rating  8892 non-null   str           
 9   Genres          8892 non-null   str           
 10  Last Updated    8892 non-null   datetime64[us]
 11  Current Ver     8888 non-null   str           
 12  Android Ver     8890 non-null   str           
dtypes: datetime64[us](1), float64(2), int64(2), str(8)
memory usage: 972.6 KB


In [47]:
apps_clean.describe(include="all")

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
count,8892,8892,8892.000000,8.892000e+03,8892,8.892000e+03,8892,8892.000000,8892,8892,8892,8888,8890
unique,8196,33,NaN,NaN,414,NaN,2,NaN,6,115,NaN,2638,31
top,ROBLOX,FAMILY,NaN,NaN,Varies with device,NaN,Free,NaN,Everyone,Tools,NaN,Varies with device,4.1 and up
freq,9,1718,NaN,NaN,1468,NaN,8279,NaN,7095,733,NaN,1258,1987
mean,NaN,NaN,4.187877,4.727764e+05,NaN,1.648965e+07,NaN,0.963155,NaN,NaN,2017-11-21 21:09:28.421052,NaN,NaN
min,NaN,NaN,1.000000,1.000000e+00,NaN,1.000000e+00,NaN,0.000000,NaN,NaN,2010-05-21 00:00:00,NaN,NaN
25%,NaN,NaN,4.000000,1.640000e+02,NaN,1.000000e+04,NaN,0.000000,NaN,NaN,2017-09-21 00:00:00,NaN,NaN
50%,NaN,NaN,4.300000,4.714500e+03,NaN,5.000000e+05,NaN,0.000000,NaN,NaN,2018-05-28 00:00:00,NaN,NaN
75%,NaN,NaN,4.500000,7.126675e+04,NaN,5.000000e+06,NaN,0.000000,NaN,NaN,2018-07-23 00:00:00,NaN,NaN
max,NaN,NaN,5.000000,7.815831e+07,NaN,1.000000e+09,NaN,400.000000,NaN,NaN,2018-08-08 00:00:00,NaN,NaN


# Save Clean Dataset

In [48]:
apps_clean.to_csv(
    "../data/processed/cleaned_googleplaystore.csv",
    index=False
)

reviews_clean.to_csv(
    "../data/processed/cleaned_googleplaystore_user_reviews.csv",
    index=False
)

# Cleaning Summary

The following cleaning steps were completed successfully:

✅ Removed corrupted records

✅ Removed duplicate rows

✅ Converted Reviews to numeric

✅ Converted Installs to numeric

✅ Converted Price to numeric

✅ Converted Last Updated to datetime

The cleaned dataset is now ready for feature engineering.

# Next Step

Proceed to:

## 03_Feature_Engineering.ipynb

where new business-focused features will be created to improve analysis and dashboard development.